In [ ]:
import math
import meep as mp
from meep import mpb
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Unit cell length is normalized to 1
num_bands = 9
k_min = -0.5
k_max = 0.5
interpolation_points = 31

k_points = [mp.Vector3(0, 0, k_min), mp.Vector3(0, 0, k_max)]
k_points = mp.interpolate(interpolation_points, k_points)

In [ ]:
# Set up materials, geometry, and simulation
geometry = [mp.Block(center=mp.Vector3(0,0,0), size=mp.Vector3(2,2,0.2), material=mp.Medium(epsilon=13))]
geometry_lattice = mp.Lattice(size=mp.Vector3(0,0,1))
default_material = mp.Medium(epsilon=1)
resolution = 32
ms = mpb.ModeSolver(num_bands=num_bands,
                    k_points=k_points,
                    geometry=geometry,
                    geometry_lattice=geometry_lattice,
                    default_material=default_material,
                    resolution=resolution)

In [ ]:
ms.run()

In [ ]:
# Check that the dielectric function looks all good

ms.output_epsilon()
md = mpb.MPBData(rectify=True, periods=3, resolution=32)
eps = ms.get_epsilon()
converted_eps = md.convert(eps)
fig1 = plt.figure()
ax1 = plt.axes(frameon=False)
plt.plot(converted_eps)
ax1.get_xaxis().set_visible(False)
ax1.set_ylabel('$\epsilon$')
plt.show()

In [ ]:
# Extract Data
freqs = ms.all_freqs
gaps = ms.gap_list
num_kpoints = len(freqs)
k=np.linspace(k_min, k_max, num_kpoints)

In [ ]:
# Plot bands
fig2, ax2 = plt.subplots()
for freq in np.transpose(freqs):
    ax2.scatter(k, freq, color='blue', facecolors='none')
    ax2.plot(k, freq, color='blue')
ax2.set_ylim([0, np.max(freqs)])
ax2.set_xlim([k[0], k[-1]])

# Plot gaps
for gap in gaps:
    if gap[0] > 1:
        ax2.fill_between(k, gap[1], gap[2], color='blue', alpha=0.2)

ax2.set_xlabel('Wave vector (ka/2$\pi$)', size=16)
ax2.set_ylabel('Frequency ($\omega$a/2$\pi$c)', size=16)
ax2.grid(True)

plt.show()

In [ ]:
# We are now interested in k-vectors with a component parallel to the surface
num_bands = 6
interpolation_points = 20
k_points = [mp.Vector3(0, 0,-0.5), mp.Vector3(0, 0, 0), mp.Vector3(0, 1.5, 0)]
k_points = mp.interpolate(interpolation_points, k_points)
geometry = [mp.Block(center=mp.Vector3(0,0,0), size=mp.Vector3(1,1,0.2), material=mp.Medium(epsilon=13))]
geometry_lattice = mp.Lattice(size=mp.Vector3(0,0,1))
default_material = mp.Medium(epsilon=1)
resolution = 30
ms = mpb.ModeSolver(
    geometry=geometry,
    geometry_lattice=geometry_lattice,
    k_points=k_points,
    default_material=default_material,
    resolution=resolution,
    num_bands=num_bands
)
ms.run_yodd()
tm_freqs = ms.all_freqs
ms.run_yeven()
te_freqs = ms.all_freqs

In [ ]:
# Plot bands
p = len(tm_freqs)
k = np.array(list(np.linspace(-0.5,0,p//2+1))[:-1] + list(np.linspace(0,1.5,p//2+1)))
fig3, ax3 = plt.subplots()
for tm_freq, te_freq in zip(np.transpose(tm_freqs), np.transpose(te_freqs)):
    ax3.scatter(k, tm_freq, color='blue')
    ax3.scatter(k, te_freq, color='red', facecolors='none')
    ax3.plot(k, tm_freq, color='blue')
    ax3.plot(k, te_freq, color='red')
ax3.set_ylim([0, 1])
ax3.set_xlim([k[0], k[-1]])

ax3.text(1, 0.3, 'TM bands', color='blue', size=15)
ax3.text(1.05, 0.6, 'TE bands', color='red', size=15)

ax3.set_xlabel('Wave vector (ka/2$\pi$)', size=16)
ax3.set_ylabel('Frequency ($\omega$a/2$\pi$c)', size=16)
ax3.grid(True)

plt.show()